# In-Class Competition: Home Credit Default Risk
## Feature Engineering and 5-Fold LightGBM Pipeline

The goal of this notebook is to predict whether a client will default on a loan (`TARGET` variable: 0 or 1) evaluated by ROC-AUC.

Approach summary:
- Fix the `DAYS_EMPLOYED == 365243` outlier.
- Add domain ratios (credit-to-income, payment rate, goods price ratio).
- Compute combinations of the three `EXT_SOURCE` features.
- Calculate frequency encodings and group averages by occupation/education.
- Train LightGBM with 5-fold stratified cross validation.
- Generate final test predictions and create the submission files.


### 1. Setup and Colab Mount
Connect Google Drive and switch to the competition folder if running on Colab.


In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # adjust path to your competition folder in drive
    %cd "/content/drive/MyDrive/competition"
    !ls
else:
    print("Running in local environment")


In [ ]:
import os
import gc
import time
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

warnings.filterwarnings('ignore')
print(f"NumPy: {np.__version__}, Pandas: {pd.__version__}, LightGBM: {lgb.__version__}")


### 2. Loading the Data


In [ ]:
current_dir = Path.cwd()
input_dir = current_dir / "input"

train_path = input_dir / "train.csv"
test_path = input_dir / "test.csv"
sample_sub_path = input_dir / "sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_sub_path)

print(f"Train shape: {train.shape}, Test shape: {test.shape}")
print("Target distribution:")
print(train['TARGET'].value_counts(normalize=True))


### 3. Feature Engineering
Here we build ratios and interactions based on the columns provided in `train.csv` and `test.csv`.


In [ ]:
def engineer_features(df):
    df = df.copy()
    
    # 365243 is a known placeholder for unemployed / pensioners in this dataset
    df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    
    # credit and income ratios
    df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
    df['INCOME_CREDIT_DIFF'] = df['AMT_INCOME_TOTAL'] - df['AMT_CREDIT']
    df['GOODS_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + 1)
    df['GOODS_CREDIT_DIFF'] = df['AMT_GOODS_PRICE'] - df['AMT_CREDIT']
    
    # per person ratios
    fam_members = df['CNT_FAM_MEMBERS'].fillna(1).replace(0, 1)
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / fam_members
    df['CREDIT_PER_PERSON'] = df['AMT_CREDIT'] / fam_members
    df['ANNUITY_PER_PERSON'] = df['AMT_ANNUITY'] / fam_members
    
    # time and age ratios
    df['DAYS_EMPLOYED_PERC'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365.25
    df['EMPLOYED_YEARS'] = -df['DAYS_EMPLOYED'] / 365.25
    df['REGISTRATION_BIRTH_RATIO'] = df['DAYS_REGISTRATION'] / df['DAYS_BIRTH']
    df['ID_PUBLISH_BIRTH_RATIO'] = df['DAYS_ID_PUBLISH'] / df['DAYS_BIRTH']
    df['PHONE_BIRTH_RATIO'] = df['DAYS_LAST_PHONE_CHANGE'] / df['DAYS_BIRTH']
    df['PHONE_EMPLOYED_RATIO'] = df['DAYS_LAST_PHONE_CHANGE'] / (df['DAYS_EMPLOYED'] - 1)
    
    # car features
    df['FLAG_OWN_CAR'] = df['OWN_CAR_AGE'].notnull().astype(int)
    df['CAR_EMPLOYED_RATIO'] = (df['OWN_CAR_AGE'] * 365) / (-df['DAYS_EMPLOYED'] + 1)
    df['CAR_BIRTH_RATIO'] = (df['OWN_CAR_AGE'] * 365) / (-df['DAYS_BIRTH'] + 1)
    
    # combinations of external source scores
    ext_sources = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    df['EXT_SOURCES_MEAN'] = df[ext_sources].mean(axis=1)
    df['EXT_SOURCES_STD'] = df[ext_sources].std(axis=1)
    df['EXT_SOURCES_MIN'] = df[ext_sources].min(axis=1)
    df['EXT_SOURCES_MAX'] = df[ext_sources].max(axis=1)
    df['EXT_SOURCES_PROD'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
    df['EXT_1_x_EXT_2'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2']
    df['EXT_2_x_EXT_3'] = df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
    df['EXT_1_x_EXT_3'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_3']
    
    df['EXT_1_over_BIRTH'] = df['EXT_SOURCE_1'] / df['DAYS_BIRTH']
    df['EXT_2_over_BIRTH'] = df['EXT_SOURCE_2'] / df['DAYS_BIRTH']
    df['EXT_3_over_BIRTH'] = df['EXT_SOURCE_3'] / df['DAYS_BIRTH']
    df['EXT_2_over_CREDIT'] = df['EXT_SOURCE_2'] / (df['AMT_CREDIT'] + 1)
    df['EXT_3_over_CREDIT'] = df['EXT_SOURCE_3'] / (df['AMT_CREDIT'] + 1)
    
    # contact flags and bureau inquiries
    df['FLAG_CONTACTS_SUM'] = df['FLAG_MOBIL'] + df['FLAG_EMP_PHONE'] + df['FLAG_WORK_PHONE']
    bureau_cols = ['AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_MON', 
                   'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR']
    df['BUREAU_ENQUIRIES_SUM'] = df[bureau_cols].sum(axis=1)
    
    return df

print("Creating features...")
train_fe = engineer_features(train)
test_fe = engineer_features(test)

# frequency encoding and group statistics
combined = pd.concat([train_fe, test_fe], axis=0, ignore_index=True)
freq_cols = ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE']
for col in freq_cols:
    freq = combined[col].value_counts(normalize=True)
    train_fe[f'{col}_FREQ'] = train_fe[col].map(freq)
    test_fe[f'{col}_FREQ'] = test_fe[col].map(freq)

for grp_col in ['NAME_EDUCATION_TYPE', 'OCCUPATION_TYPE']:
    for num_col in ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'EXT_SOURCE_2']:
        grp_stats = combined.groupby(grp_col)[num_col].agg(['mean']).reset_index()
        grp_stats.columns = [grp_col, f'{grp_col}_{num_col}_MEAN']
        train_fe = train_fe.merge(grp_stats, on=grp_col, how='left')
        test_fe = test_fe.merge(grp_stats, on=grp_col, how='left')
        train_fe[f'{num_col}_RATIO_{grp_col}'] = train_fe[num_col] / (train_fe[f'{grp_col}_{num_col}_MEAN'] + 1)
        test_fe[f'{num_col}_RATIO_{grp_col}'] = test_fe[num_col] / (test_fe[f'{grp_col}_{num_col}_MEAN'] + 1)

print(f"Total features created: {train_fe.shape[1] - 2}")


### 4. Categorical Encoding


In [ ]:
ignore_cols = ['SK_ID_CURR', 'TARGET']
features = [c for c in train_fe.columns if c not in ignore_cols]
cat_cols = train_fe[features].select_dtypes(include=['object', 'category']).columns.tolist()

for c in cat_cols:
    le = LabelEncoder()
    unique_vals = list(train_fe[c].astype(str).unique()) + list(test_fe[c].astype(str).unique())
    le.fit(unique_vals)
    train_fe[c] = le.transform(train_fe[c].astype(str)).astype('category')
    test_fe[c] = le.transform(test_fe[c].astype(str)).astype('category')

X = train_fe[features]
y = train['TARGET'].values
X_test = test_fe[features]
test_ids = test['SK_ID_CURR']
print(f"Data ready: {X.shape[1]} features, {len(X)} rows")


### 5. Model Training (5-Fold Stratified Cross Validation)
We use 5 folds with early stopping to prevent overfitting and get out-of-fold predictions.


In [ ]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))
feature_importances = pd.DataFrame(index=features)

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 10.0,
    'random_state': 42,
    'n_estimators': 1500,
    'verbose': -1,
    'n_jobs': -1
}

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"Fold {fold + 1} / {n_splits}")
    X_tr, y_tr = X.iloc[train_idx], y[train_idx]
    X_va, y_va = X.iloc[val_idx], y[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    callbacks = [
        lgb.early_stopping(stopping_rounds=100, verbose=False),
        lgb.log_evaluation(period=200)
    ]
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_tr, y_tr), (X_va, y_va)],
        eval_names=['train', 'valid'],
        categorical_feature=cat_cols,
        callbacks=callbacks
    )
    
    val_pred = model.predict_proba(X_va)[:, 1]
    oof_predictions[val_idx] = val_pred
    
    fold_auc = roc_auc_score(y_va, val_pred)
    fold_scores.append(fold_auc)
    print(f"Fold {fold + 1} validation AUC: {fold_auc:.5f} (best iteration: {model.best_iteration_})")
    
    test_pred_fold = model.predict_proba(X_test)[:, 1]
    test_predictions += test_pred_fold / n_splits
    feature_importances[f'fold_{fold+1}'] = model.feature_importances_

overall_cv_auc = roc_auc_score(y, oof_predictions)
print("-" * 50)
print(f"Overall OOF ROC-AUC: {overall_cv_auc:.5f}")
print(f"Mean fold score: {np.mean(fold_scores):.5f} +/- {np.std(fold_scores):.5f}")
print("-" * 50)


### 6. Feature Importance


In [ ]:
feature_importances['mean_importance'] = feature_importances.mean(axis=1)
top20 = feature_importances['mean_importance'].sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(x=top20.values, y=top20.index)
plt.title('Top 20 Features by Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


### 7. Save Submission and Create ZIP for Honors Submission


In [ ]:
output_dir = current_dir / "output"
os.makedirs(output_dir, exist_ok=True)
sub_file = output_dir / "submission.csv"

submission = pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET': test_predictions
})
submission.to_csv(sub_file, index=False)
print(f"Saved submission file to: {sub_file}")
print(f"Submission shape: {submission.shape}")
print(submission.head())


In [ ]:
# create zip archive for honors requirement
nb_name = "champion_solution.ipynb"
zip_name = "champion_solution.zip"

nb_file = current_dir / nb_name
zip_file = current_dir / zip_name

with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    if nb_file.exists():
        zf.write(nb_file, arcname=nb_name)

with zipfile.ZipFile(current_dir / "tutorial.zip", 'w', zipfile.ZIP_DEFLATED) as zf:
    if nb_file.exists():
        zf.write(nb_file, arcname="tutorial.ipynb")

print("Created zip files for submission")


In [ ]:
# download files in colab
if IN_COLAB:
    from google.colab import files
    files.download(str(sub_file))
    files.download(str(zip_file))
else:
    print("Files ready in the output folder.")


### 8. References
Code structure and feature engineering ideas adapted from:
- Kaggle Home Credit Default Risk discussions and public notebooks.
- Aguiar, "LightGBM with Simple Features".
- Will Koehrsen, "Home Credit Default Risk Baseline & EDA".
